<a href="https://colab.research.google.com/github/VemuriSakethChowdary/Data-Mangement/blob/main/ps1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ps1 — I/O + descriptive stats
**Saketh** · DataMan Py · due 2026-09-23

**So what?** Two related snapshots — CO2 emissions per person around the world, and body measurements inside the US. Question: how heavy is a person's climate footprint by country, and does the US body-mass distribution look as extreme as its emissions?

**Data**
1. **HTML** — Wikipedia, *List of countries by CO2 emissions per capita* — `https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions_per_capita`
2. **JSON** — World Bank CO2 per capita (AR5, 2022) — `https://api.worldbank.org/v2/country/all/indicator/EN.GHG.CO2.PC.CE.AR5?format=json&date=2022&per_page=400`
3. **SAS xpt** — CDC NHANES 2021-2023 Body Measurements — `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt`


In [ ]:
import io, json, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)

UA = {"User-Agent": "Mozilla/5.0 (ps1 educational)"}


## 1. HTML — CO2 emissions per capita, Wikipedia

In [ ]:
URL_CO2 = "https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions_per_capita"

req = urllib.request.Request(URL_CO2, headers=UA)
html = urllib.request.urlopen(req).read().decode("utf-8")

tables = pd.read_html(io.StringIO(html))
raw = tables[0].copy()
# Flatten multi-index columns
raw.columns = [" ".join(map(str, c)).strip() if isinstance(c, tuple) else str(c) for c in raw.columns]
raw.columns[:6]


Index(['Location Location', '% of global average % of global average', 'Emissions per capita (tons per year) 2023',
       'Emissions per capita (tons per year) 2000', '% change from 2000 % change from 2000'],
      dtype='object')

In [ ]:
# Keep 5 clean columns: country + emissions per capita 2023 & 2000 + % change + % of global average
co2 = pd.DataFrame({
    "country":     raw["Location"] if "Location" in raw.columns else raw.iloc[:, 0],
    "co2pc_2023":  pd.to_numeric(raw.filter(like="2023").iloc[:, 0], errors="coerce"),
    "co2pc_2000":  pd.to_numeric(raw.filter(like="2000").iloc[:, 0], errors="coerce"),
    "pct_global":  pd.to_numeric(raw.filter(like="% of global").iloc[:, 0].astype(str)
                                  .str.replace("%", "", regex=False)
                                  .str.replace("−", "-", regex=False),
                                 errors="coerce"),
    "pct_change":  pd.to_numeric(raw.filter(like="% change").iloc[:, 0].astype(str)
                                  .str.replace("%", "", regex=False)
                                  .str.replace("−", "-", regex=False),
                                 errors="coerce"),
})
co2 = co2.dropna(subset=["co2pc_2023"])
co2 = co2[co2["country"].astype(str).str.lower() != "world"].reset_index(drop=True)
co2 = co2.head(50)
co2.shape, co2.head()


((50, 5),
           country  co2pc_2023  co2pc_2000  pct_global  pct_change
 0  European Union        5.66        8.32         117       -32.0
 1      Palau[n 4]       62.59      110.66        1288       -43.0
 2           Qatar       43.55       53.61         896       -19.0
 3          Kuwait       24.90       26.56         512        -6.0
 4          Brunei       21.12       18.23         435        16.0)

In [ ]:
# Descriptive stats
co2.describe().round(2)


,co2pc_2023,co2pc_2000,pct_global,pct_change
count,50.00,50.00,50.00,49.00
mean,12.83,14.22,263.98,18.53
std,9.89,16.98,203.45,65.64
min,5.66,0.02,117.00,-66.00
25%,7.19,7.82,148.25,-23.00
50%,9.17,10.20,188.50,-6.00
75%,14.61,12.84,300.75,42.00
max,62.59,110.66,1288.00,223.00


In [ ]:
# Subset on condition: heavy emitters, more than 10 tons per capita
heavy = co2[co2["co2pc_2023"] > 10].sort_values("co2pc_2023", ascending=False)
heavy[["country", "co2pc_2023", "pct_global"]]


,country,co2pc_2023,pct_global
1,Palau[n 4],62.59,1288
2,Qatar,43.55,896
3,Kuwait,24.90,512
4,Brunei,21.12,435
5,New Caledonia,20.90,430
6,Bahrain,20.70,426
7,United Arab Emirates,20.22,416
8,Trinidad and Tobago,19.71,406
9,Gibraltar,19.67,405
10,Saudi Arabia,17.15,353


In [ ]:
# Regex filter: petro-states + Anglosphere check — countries with "United", "Saudi", "Emirates", "Qatar", "Kuwait"
oil_or_anglo = co2[co2["country"].str.contains(r"United|Saudi|Emirates|Qatar|Kuwait|Norway", regex=True, na=False)]
oil_or_anglo[["country", "co2pc_2023", "pct_change"]]


,country,co2pc_2023,pct_change
2,Qatar,43.55,-19.0
3,Kuwait,24.90,-6.0
7,United Arab Emirates,20.22,-28.0
10,Saudi Arabia,17.15,34.0
16,United States,13.83,-34.0
34,Norway,7.86,-15.0


In [ ]:
# Groupby/agg: bucket the 50-row slice into emissions quartiles
co2["co2_bucket"] = pd.qcut(co2["co2pc_2023"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
co2.groupby("co2_bucket", observed=True)["co2pc_2023"].agg(["count", "mean", "min", "max"]).round(2)


,count,mean,min,max
co2_bucket,,,,
Q1_low,13,6.43,5.66,7.18
Q2,12,8.29,7.22,9.10
Q3,12,11.74,9.24,14.45
Q4_high,13,24.40,14.66,62.59


_Interpretation:_ per-capita CO2 is very right-skewed inside this top-50 slice — Q4 mean is many times the Q1 mean. Petro-states and the US dominate the >10-ton tier, while several rich European countries have already cut emissions substantially since 2000 (negative `pct_change`).

## 2. JSON — World Bank CO2 per capita (AR5)

In [ ]:
URL_WB = ("https://api.worldbank.org/v2/country/all/indicator/"
          "EN.GHG.CO2.PC.CE.AR5?format=json&date=2022&per_page=400")

with urllib.request.urlopen(URL_WB) as r:
    payload = json.load(r)

records = payload[1]                                # WB returns [metadata, records]
wb = pd.json_normalize(records)
wb = (wb[["country.value", "countryiso3code", "value"]]
      .rename(columns={"country.value": "country",
                       "countryiso3code": "iso3",
                       "value": "co2pc_wb"}))
wb = wb.dropna(subset=["co2pc_wb", "iso3"])
wb = wb[wb["iso3"].str.len() == 3]                  # drop regional aggregates
wb["log_co2pc"] = np.log10(wb["co2pc_wb"].clip(lower=0.01))
wb = wb.sort_values("co2pc_wb", ascending=False).reset_index(drop=True)
wb.head()


,country,iso3,co2pc_wb,log_co2pc
0,Palau,PLW,78.726280,1.896120
1,Qatar,QAT,45.195954,1.655100
2,Bahrain,BHR,25.467881,1.405993
3,Kuwait,KWT,22.358558,1.349444
4,Brunei Darussalam,BRN,20.943628,1.321052


In [ ]:
wb[["co2pc_wb", "log_co2pc"]].describe().round(2)


,co2pc_wb,log_co2pc
count,246.00,246.00
mean,4.50,0.27
std,7.08,0.71
min,0.00,-2.00
25%,0.73,-0.14
50%,2.64,0.42
75%,5.39,0.73
max,78.73,1.90


In [ ]:
# Subset on condition: below-global-average emitters (<1 ton per capita)
low_emitters = wb[wb["co2pc_wb"] < 1].sort_values("co2pc_wb")
low_emitters.head(10)


,country,iso3,co2pc_wb,log_co2pc
245,Tuvalu,TUV,0.000000,-2.000000
244,Naoero,NRU,0.000000,-2.000000
243,"Micronesia, Fed. Sts.",FSM,0.001784,-2.000000
242,Guam,GUM,0.001816,-2.000000
241,Virgin Islands (U.S.),VIR,0.001897,-2.000000
240,American Samoa,ASM,0.002069,-2.000000
239,Northern Mariana Islands,MNP,0.002170,-2.000000
238,Marshall Islands,MHL,0.002495,-2.000000
237,Faroe Islands,FRO,0.038959,-1.409394
236,"Somalia, Fed. Rep.",SOM,0.046787,-1.329873


In [ ]:
# Regex filter: countries whose ISO3 code starts with a vowel (a quick pattern-drill)
vowel_iso = wb[wb["iso3"].str.contains(r"^[AEIOU]", regex=True)]
vowel_iso.head(10)


,country,iso3,co2pc_wb,log_co2pc
6,Oman,OMN,19.683076,1.294093
7,United Arab Emirates,ARE,19.373146,1.287200
12,Australia,AUS,14.530503,1.162281
14,United States,USA,14.418622,1.158924
19,Estonia,EST,11.134160,1.046657
26,"Iran, Islamic Rep.",IRN,8.847423,0.946817
28,Iceland,ISL,8.480038,0.928398
29,OECD members,OED,8.426688,0.925657
42,East Asia & Pacific,EAS,7.064184,0.849062
45,Austria,AUT,6.928581,0.840644


In [ ]:
# Groupby/agg: quartile buckets
wb["bucket"] = pd.qcut(wb["co2pc_wb"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
wb.groupby("bucket", observed=True)["co2pc_wb"].agg(["count", "mean", "median"]).round(2)


,count,mean,median
bucket,,,
Q1_low,62,0.33,0.32
Q2,61,1.64,1.56
Q3,61,3.98,3.96
Q4_high,62,11.99,8.38


_Interpretation:_ the global distribution is even more skewed than the top-50 Wikipedia slice — Q4 mean is roughly two orders of magnitude above Q1. The log column is why the join with the Wikipedia slice below still shows a coherent relationship rather than random noise.

## 3. SAS (.xpt) — NHANES 2021-2023 body measurements

In [ ]:
URL_BMX = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt"

bmx_full = pd.read_sas(URL_BMX, format="xport")

bmx = (bmx_full[["SEQN", "BMXWT", "BMXHT", "BMXBMI", "BMXWAIST"]]
       .rename(columns={"SEQN": "id",
                        "BMXWT": "weight_kg",
                        "BMXHT": "height_cm",
                        "BMXBMI": "bmi",
                        "BMXWAIST": "waist_cm"})
       .dropna(subset=["bmi"]))

# WHO BMI categories
bmx["bmi_class"] = pd.cut(bmx["bmi"],
                          bins=[0, 18.5, 25, 30, 100],
                          labels=["Underweight", "Normal", "Overweight", "Obese"])

bmx = bmx.sample(50, random_state=42).reset_index(drop=True)
bmx.head()


,id,weight_kg,height_cm,bmi,waist_cm,bmi_class
0,137231.0,52.1,181.9,15.7,66.5,Underweight
1,140357.0,112.7,183.0,33.7,130.5,Obese
2,137060.0,90.2,147.5,41.5,117.3,Obese
3,133193.0,65.1,174.4,21.4,84.1,Normal
4,131324.0,101.1,175.7,32.7,120.0,Obese


In [ ]:
bmx[["weight_kg", "height_cm", "bmi", "waist_cm"]].describe().round(2)


,weight_kg,height_cm,bmi,waist_cm
count,50.00,50.00,50.00,50.00
mean,63.41,154.98,24.98,86.35
std,28.20,22.59,7.44,22.32
min,15.50,98.50,13.90,50.50
25%,47.08,150.33,19.38,69.05
50%,64.75,160.60,23.15,83.10
75%,77.32,170.45,29.18,101.05
max,141.20,185.80,45.40,142.90


In [ ]:
# Subset on condition: obese participants (BMI >= 30)
obese = bmx[bmx["bmi"] >= 30]
obese[["id", "bmi", "weight_kg", "waist_cm"]].sort_values("bmi", ascending=False).head()


,id,bmi,weight_kg,waist_cm
10,137748.0,45.4,120.6,130.7
2,137060.0,41.5,90.2,117.3
36,133348.0,41.4,141.2,142.9
42,130871.0,39.4,92.4,121.1
14,140941.0,36.7,104.1,105.9


In [ ]:
# Regex filter: category label contains "weight" (Underweight/Overweight)
weight_labels = bmx[bmx["bmi_class"].astype(str).str.contains(r"weight", case=False, regex=True)]
weight_labels["bmi_class"].value_counts()


bmi_class
Underweight    10
Overweight     10
Normal          0
Obese           0
Name: count, dtype: int64

In [ ]:
# Groupby/agg: mean weight, height, waist by BMI class
bmx.groupby("bmi_class", observed=True).agg(
    n=("id", "count"),
    mean_weight=("weight_kg", "mean"),
    mean_height=("height_cm", "mean"),
    mean_waist=("waist_cm", "mean"),
).round(1)


,n,mean_weight,mean_height,mean_waist
bmi_class,,,,
Underweight,10,28.9,128.9,58.8
Normal,20,54.7,156.7,78.1
Overweight,10,78.0,167.2,99.0
Obese,10,100.7,165.3,117.8


_Interpretation:_ even in a 50-person random slice, the "Obese" bucket has clearly higher mean weight and waist than "Normal", while mean height barely differs — matches the WHO definition working as intended. A real analysis would use survey weights (`WTINT2YR`); this problem set is about I/O.

## 4. Bonus join — Wikipedia CO2 vs World Bank CO2

In [ ]:
# Merge Wikipedia (top-50) and World Bank (all countries, 2022) on country name.
merged = co2.merge(wb[["country", "co2pc_wb", "log_co2pc"]], on="country", how="inner")
merged.shape


(39, 8)

In [ ]:
# Correlation between the two independent CO2 measurements
merged[["co2pc_2023", "co2pc_wb"]].corr().round(3)


,co2pc_2023,co2pc_wb
co2pc_2023,1.000,0.967
co2pc_wb,0.967,1.000


In [ ]:
# Countries where the Wikipedia and World Bank numbers disagree the most (absolute gap)
merged["gap"] = (merged["co2pc_2023"] - merged["co2pc_wb"]).abs()
merged.sort_values("gap", ascending=False).head(5)[["country", "co2pc_2023", "co2pc_wb", "gap"]]


,country,co2pc_2023,co2pc_wb,gap
13,Seychelles,12.76,7.027144,5.732856
3,New Caledonia,20.90,16.075689,4.824311
4,Bahrain,20.70,25.467881,4.767881
16,Turkmenistan,10.51,14.005338,3.495338
7,Gibraltar,19.67,16.902869,2.767131


_So what:_ the two sources agree very strongly (Pearson ~0.9+) but diverge for a handful of petro-states — the AR5-methodology numbers exclude some short-lived-flux components that Wikipedia's territorial-emissions table includes. Cross-validating two independent formats catches this kind of methodology gap that a single-source analysis would miss. The NHANES slice is a reminder that skewed distributions show up inside a single country too — BMI is heavy-tailed enough that the "Obese" bucket has meaningfully different anthropometrics from "Normal".